<a href="https://colab.research.google.com/github/mohamed2ayman/Claude/blob/main/CodeBasedGrading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

#Load new Variable
from dotenv import load_dotenv
load_dotenv()

# Create API Client
import os
from anthropic import Anthropic

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

client = Anthropic()
model = "claude-sonnet-4-5"

In [7]:
# Add user message, while will tak list of messages and text. It's reason is to append messages together and save it to history.

def add_user_message(messages, text):
  user_message = {"role":"user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role": "assistant", "content": text}
  messages.append(assistant_message)


def chat(messages, system=None, temperature =1.0, stop_sequences=None ):
  params = {
      "model": model,
      "max_tokens": 1000,
      "messages": messages,
      "temperature": temperature
  }

  if system:
    params["system"] = system

  if stop_sequences:
    params["stop_sequences"] = stop_sequences

  message = client.messages.create(**params)

  return message.content[0].text

In [8]:
# Creating an Evaluation Dataset for Prompt Evaluation

import json

def generate_dataset():
  prompt= """

Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python,
JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python,
JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "Format": "json" or "python" or "regex"
    "Solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "'''json")
  text = chat(messages, stop_sequences=["'''"])
  return json.loads(text)


In [10]:
# Dumping the dataset evaluation in a JSON file
import json
dataset= generate_dataset()

with open("dataset.json", "w") as f:
  json.dump(dataset, f, indent=2)

In [11]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt= f"""

Please solve the following task:

{test_case["task"]}


* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

  messages=[]
  add_user_message(messages, prompt)
  add_assistant_message(messages, "'''code")
  output = chat(messages, stop_sequences= ["'''"])
  return output


In [13]:
def grade_by_model(test_case, output):
  # Create evaluation prompt
    eval_prompt =f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Original Task:

    <task>
    Task: {test_case["task"]}
    </task>

    Solution to evaluate:

    <solution>
    Solution: {output}
    </solution>

    Criteria you should use to evaluate the solution:

    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10


    Respond with JSON. Keep your response concise and direct.
    Example response shape:

   {{

      "strengths": string[],
      "weaknesses": string[],
      "reasoning": string,
      "score": number
   }}

      """



    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [14]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)

    elif format == "python":
         return validate_python(response)

    else:
         return validate_regex(response)

In [15]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the resullt"""
  output = run_prompt(test_case)

  # TODO -Grading
  model_grade = grade_by_model(test_case, output)
  model_score = model_grade["score"]
  reasoning = model_grade["reasoning"]


  sytnax_score = grade_syntax(output, test_case)

  score= (model_score + sytnax_score)/2


  return {

          "output": output,
          "test_case": test_case,
          "score": score,
          "reasoning": reasoning
  }


In [16]:

from statistics import mean

def run_eval(dataset):
  """Loads the dataset and calls run_test_case for each test case"""
  results = []
  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result["score"] for result in results])
  print(f"Average score: {average_score}")

  return results

In [17]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.5


In [18]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n^(?!.*\\\\.\\\\.)(?!.*\\\\.-)(?!.*-\\\\.)([a-z0-9])([a-z0-9.-]{1,61})([a-z0-9])$\n",
    "test_case": {
      "task": "Create a regex pattern that validates AWS S3 bucket names according to AWS naming rules: 3-63 characters long, only lowercase letters, numbers, hyphens, and periods, must start and end with a letter or number, and cannot have consecutive periods or period-hyphen combinations.",
      "format": "regex",
      "solution_criteria": "Pattern must reject invalid bucket names (uppercase letters, underscores, <3 or >63 chars, starting/ending with hyphen/period, consecutive periods like '..' or '.-' or '-.') and accept valid bucket names with lowercase alphanumeric characters, single periods and hyphens in appropriate positions."
    },
    "score": 8.5,
    "reasoning": "The regex correctly handles most core requirements including character restrictions, start/end validation, and forbidden consecutive character patterns. However, it misses the IP addres